### **Alineamiento multimodal, aprendizaje contrastivo y recuperación semántica con re-ranking**


Este cuaderno se sitúa en una etapa avanzada de la secuencia del curso y marca el tránsito desde los modelos de **fusión multimodal profunda** hacia los modelos de **alineamiento multimodal en espacio compartido**. Su propósito no es únicamente integrar modalidades dentro de un clasificador, sino estudiar cómo texto e imagen pueden ser proyectados a una geometría semántica común, donde la cercanía entre representaciones tenga valor operativo para tareas de recuperación, emparejamiento y refinamiento posterior.

La tesis central del cuaderno es que, en escenarios multimodales complejos, no siempre resulta óptimo resolver todo mediante una sola arquitectura de fusión temprana o profunda. En muchos casos, una estrategia en dos niveles ofrece mayor claridad metodológica y mejor escalabilidad conceptual:

**primero**, aprender un mecanismo de **alineamiento contrastivo** entre modalidades;
**después**, aplicar un procedimiento de **re-ranking** o interacción cruzada solo sobre los candidatos más ambiguos o informativos.

Este principio organiza una parte importante de la investigación contemporánea en sistemas multimodales, y constituye un puente directo hacia arquitecturas y marcos de preentrenamiento más avanzados.


El cuaderno trabaja sobre cinco ideas mayores:

- **Alineamiento multimodal.**
Texto e imagen no se entienden aquí solo como fuentes que deben fusionarse, sino como dominios representacionales distintos que pueden ser coordinados mediante proyecciones a un espacio semántico común.

- **Bi-encoders contrastivos.**
Cada modalidad se codifica de forma independiente y luego se optimiza para que los pares correctos se acerquen y los pares incorrectos se separen. Esta formulación permite estudiar recuperación multimodal de manera eficiente y conceptualmente limpia.

- **Negativos difíciles y semi-duros.**
La calidad del aprendizaje contrastivo depende de la dificultad de los ejemplos negativos. El cuaderno enfatiza que los negativos no deben ser meramente aleatorios, sino suficientemente cercanos para obligar al modelo a aprender fronteras semánticas más finas.

- **Retrieval multimodal.**
La evaluación deja de centrarse exclusivamente en clasificación y pasa a considerar tareas de recuperación, donde el objetivo es ordenar candidatos relevantes según similitud entre modalidades.

- **Re-ranking con interacción cruzada.**
El alineamiento inicial producido por un bi-encoder puede ser refinado por un segundo modelo que examina de manera más detallada la interacción entre la consulta y los candidatos recuperados. Este segundo nivel introduce un compromiso explícito entre eficiencia y precisión.

Este cuaderno cumple una función estratégica: mostrar que la historia reciente de la multimodalidad no puede entenderse solo como una sucesión de arquitecturas de fusión cada vez más profundas. También debe entenderse como una transición hacia modelos que combinan:

- representaciones compartidas,
- objetivos contrastivos,
- mecanismos de recuperación,
- y refinamiento posterior de candidatos.

Por eso, este cuaderno no sustituye a los anteriores, sino que reinterpreta su desarrollo: después de aprender a fusionar, ahora corresponde aprender a alinear, recuperar y refinar.


#### **Orientación de uso**

**Ejecución en entorno CPU**

Cuando el trabajo se realice en CPU, conviene adoptar una configuración de desarrollo controlado y de bajo costo computacional. En ese escenario, se recomienda activar `fast_dev_run=True`, trabajar con subconjuntos reducidos del dataset y limitar el entrenamiento a una sola época por modelo. El objetivo, en este caso, no es maximizar desempeño, sino verificar consistencia del pipeline, estabilidad de la implementación y coherencia de las métricas antes de escalar el experimento.

**Ejecución en entorno GPU**

Cuando se disponga de GPU, el cuaderno puede utilizarse en un régimen experimental más cercano a una práctica de investigación. En ese contexto, se recomienda desactivar `fast_dev_run`, ampliar el tamaño del subconjunto de datos, incrementar el número de épocas y ejecutar comparaciones controladas o ablacións. Esto permite observar con mayor claridad el comportamiento relativo entre variantes del modelo, el efecto de los negativos difíciles y la contribución del re-ranking dentro del pipeline multimodal.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 0. Setup

import os
import math
import random
import copy
from dataclasses import dataclass, asdict
from typing import Dict, Optional, List, Tuple, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Si estás en un entorno nuevo, descomenta:
# %pip install -q -U datasets transformers accelerate scikit-learn pillow matplotlib tqdm

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, AutoImageProcessor
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

SEED = 42

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

def resolve_device(preference: str = "auto") -> torch.device:
    preference = preference.lower().strip()
    if preference == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if preference == "cuda":
        if torch.cuda.is_available():
            return torch.device("cuda")
        print("CUDA no disponible. Se usará CPU.")
        return torch.device("cpu")
    if preference == "cpu":
        return torch.device("cpu")
    raise ValueError("device_preference debe ser 'auto', 'cpu' o 'cuda'.")

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 1. Configuración

@dataclass
class Config:
    dataset_name: str = "jxie/flickr8k"

    max_images_train: int = 800
    max_images_val: int = 200
    max_images_test: int = 200

    max_text_len: int = 40
    batch_size: int = 16

    epochs_biencoder: int = 3
    epochs_reranker: int = 2

    text_model_name: str = "distilbert-base-uncased"
    embed_dim: int = 256
    dropout: float = 0.1
    temperature: float = 0.07

    lr_biencoder: float = 2e-4
    lr_reranker: float = 2e-4
    weight_decay: float = 1e-4

    device_preference: str = "auto"
    num_workers: int = 0

    negatives_for_reranker: str = "semihard"   # random | semihard
    topk_for_reranking: int = 10

    run_biencoder: bool = True
    run_reranker: bool = True
    fast_dev_run: bool = False

    save_dir: str = "./cuaderno6_final_runs"

CFG = Config()

if CFG.fast_dev_run:
    CFG.max_images_train = 120
    CFG.max_images_val = 60
    CFG.max_images_test = 60
    CFG.batch_size = 8
    CFG.epochs_biencoder = 1
    CFG.epochs_reranker = 1
    CFG.topk_for_reranking = 5

DEVICE = resolve_device(CFG.device_preference)
os.makedirs(CFG.save_dir, exist_ok=True)

print("DEVICE:", DEVICE)
CFG

#### **1. Carga de datos**

En este cuaderno se trabaja con **Flickr8k**  ya que permite articular, en una misma base experimental, varios niveles de análisis:

* **retrieval imagen–texto**, donde una imagen actúa como consulta y el sistema debe recuperar descripciones relevantes,
* **retrieval texto–imagen**, donde una descripción textual se usa para recuperar la imagen correspondiente,
* **tareas de match / mismatch**, orientadas a decidir si un par imagen–caption constituye o no una asociación válida,
* **estrategias de hard negative mining sobre captions**, fundamentales para estudiar cómo la dificultad de los ejemplos negativos afecta la calidad del alineamiento representacional.


In [ ]:
dataset = load_dataset(CFG.dataset_name)
dataset

In [ ]:
def build_flickr8k_splits(dataset, max_images_train=800, max_images_val=200, max_images_test=200):
    train_raw = dataset["train"][:max_images_train]
    val_raw = dataset["validation"][:max_images_val]
    test_raw = dataset["test"][:max_images_test]

    def explode_split(split_dict, split_name):
        rows = []
        caption_cols = sorted([k for k in split_dict.keys() if k.startswith("caption_")])

        for i in range(len(split_dict["image"])):
            image = split_dict["image"][i]
            image_id = f"{split_name}_{i}"
            captions = [split_dict[c][i] for c in caption_cols]

            for cap in captions:
                rows.append({
                    "split": split_name,
                    "image_id": image_id,
                    "image": image,
                    "caption": cap,
                })

        return pd.DataFrame(rows)

    train_df = explode_split(train_raw, "train")
    val_df = explode_split(val_raw, "val")
    test_df = explode_split(test_raw, "test")
    return train_df, val_df, test_df

train_df, val_df, test_df = build_flickr8k_splits(
    dataset,
    CFG.max_images_train,
    CFG.max_images_val,
    CFG.max_images_test
)

print("train rows:", len(train_df))
print("val rows:", len(val_df))
print("test rows:", len(test_df))
train_df.head()

#### **2. Exploración y visualización**

In [ ]:
sample_rows = train_df.sample(3, random_state=SEED)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (_, row) in zip(axes, sample_rows.iterrows()):
    ax.imshow(row["image"])
    ax.set_title(row["caption"][:80], fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
caption_lengths = train_df["caption"].str.split().apply(len)

print(caption_lengths.describe())

plt.figure(figsize=(7,4))
plt.hist(caption_lengths, bins=20)
plt.title("Distribución de longitud de captions")
plt.xlabel("número de palabras")
plt.ylabel("frecuencia")
plt.show()

#### **3. Negativos aleatorios y semi-duros**

La construcción de ejemplos negativos constituye un componente metodológico central en este cuaderno, porque la calidad del aprendizaje contrastivo depende en gran medida de **qué tan informativos** sean los pares incorrectos utilizados durante el entrenamiento. No todos los negativos contribuyen del mismo modo: algunos resultan demasiado evidentes y, por tanto, apenas fuerzan al modelo a refinar sus representaciones; otros, en cambio, introducen una dificultad semántica suficiente como para volver más exigente el proceso de alineamiento.

**Negativo aleatorio**

Se denomina **negativo aleatorio** al caption tomado de una imagen distinta, seleccionado sin ningún criterio adicional de proximidad semántica. Este tipo de negativo cumple una función básica: garantiza que el par imagen–texto no corresponde. Sin embargo, desde un punto de vista formativo y representacional, suele ser un ejemplo relativamente fácil, ya que muchas veces la discrepancia entre el positivo y el negativo es demasiado evidente para el modelo.

**Negativo semi-duro**

Se denomina **negativo semi-duro** al caption que, aun perteneciendo a otra imagen, presenta una cercanía léxica o semántica superficial respecto del caption positivo. En este caso, el modelo ya no puede apoyarse en diferencias triviales, sino que debe aprender rasgos más finos de correspondencia multimodal para distinguir entre asociaciones correctas e incorrectas.

Para aproximar este tipo de dificultad intermedia, el cuaderno utiliza una estrategia basada en dos herramientas sencillas pero eficaces:

* **TF-IDF**, como representación vectorial de los captions en función de su contenido léxico;
* **vecinos cercanos con distancia coseno**, para identificar captions próximos entre sí dentro de ese espacio textual.

La lógica es la siguiente: dado un caption positivo, se busca otro caption que sea relativamente cercano en el espacio TF-IDF, pero que pertenezca a una imagen diferente. Ese candidato se utiliza como negativo semi-duro. Así, el sistema enfrenta ejemplos incorrectos que no son arbitrarios, sino plausibles desde el punto de vista superficial del lenguaje.


In [ ]:
def build_random_negative_map(df: pd.DataFrame):
    image_ids = df["image_id"].tolist()
    neg_for = {}
    for i in range(len(df)):
        candidates = [k for k in range(len(df)) if image_ids[k] != image_ids[i]]
        neg_for[i] = random.choice(candidates)
    return neg_for

def build_semihard_negative_map(df: pd.DataFrame):
    captions = df["caption"].tolist()
    image_ids = df["image_id"].tolist()

    tfidf = TfidfVectorizer(max_features=10000, stop_words="english")
    X = tfidf.fit_transform(captions)

    nn_model = NearestNeighbors(n_neighbors=20, metric="cosine")
    nn_model.fit(X)
    _, indices = nn_model.kneighbors(X)

    negative_for = {}
    for i in range(len(captions)):
        src_img = image_ids[i]
        neg_idx = None
        for j in indices[i, 1:]:
            if image_ids[j] != src_img:
                neg_idx = int(j)
                break
        if neg_idx is None:
            candidates = [k for k in range(len(captions)) if image_ids[k] != src_img]
            neg_idx = random.choice(candidates)
        negative_for[i] = neg_idx
    return negative_for

train_neg_map_random = build_random_negative_map(train_df)
train_neg_map_semihard = build_semihard_negative_map(train_df)

probe_idx = 0
print("POSITIVO")
print(train_df.iloc[probe_idx]["caption"])
print()
print("NEGATIVO ALEATORIO")
print(train_df.iloc[train_neg_map_random[probe_idx]]["caption"])
print()
print("NEGATIVO SEMI-DURO")
print(train_df.iloc[train_neg_map_semihard[probe_idx]]["caption"])

#### **6. Tokenizador y preprocesador visual**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG.text_model_name)
image_processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")

#### **7. Dataset y dataloaders**

In [ ]:
class FlickrPairDataset(Dataset):
    def __init__(self, df: pd.DataFrame, negative_map: Optional[Dict[int, int]] = None, mode: str = "biencoder"):
        self.df = df.reset_index(drop=True)
        self.negative_map = negative_map
        self.mode = mode

    def __len__(self):
        if self.mode == "biencoder":
            return len(self.df)
        if self.mode == "reranker":
            return len(self.df) * 2
        raise ValueError("mode debe ser 'biencoder' o 'reranker'")

    def __getitem__(self, idx):
        if self.mode == "biencoder":
            row = self.df.iloc[idx]
            return {
                "image": row["image"],
                "caption": row["caption"],
                "label": 1,
                "image_id": row["image_id"],
            }

        pos_idx = idx // 2
        is_positive = (idx % 2 == 0)

        if is_positive:
            row = self.df.iloc[pos_idx]
            return {
                "image": row["image"],
                "caption": row["caption"],
                "label": 1,
                "image_id": row["image_id"],
            }

        neg_idx = self.negative_map[pos_idx]
        row_img = self.df.iloc[pos_idx]
        row_txt = self.df.iloc[neg_idx]
        return {
            "image": row_img["image"],
            "caption": row_txt["caption"],
            "label": 0,
            "image_id": row_img["image_id"],
        }

def collate_biencoder(batch):
    captions = [x["caption"] for x in batch]
    images = [x["image"] for x in batch]

    tok = tokenizer(
        captions,
        padding=True,
        truncation=True,
        max_length=CFG.max_text_len,
        return_tensors="pt"
    )
    pixel_values = image_processor(images=images, return_tensors="pt")["pixel_values"]

    return {
        "input_ids": tok["input_ids"],
        "attention_mask": tok["attention_mask"],
        "pixel_values": pixel_values,
        "captions": captions,
    }

def collate_reranker(batch):
    captions = [x["caption"] for x in batch]
    images = [x["image"] for x in batch]
    labels = torch.tensor([x["label"] for x in batch], dtype=torch.long)

    tok = tokenizer(
        captions,
        padding=True,
        truncation=True,
        max_length=CFG.max_text_len,
        return_tensors="pt"
    )
    pixel_values = image_processor(images=images, return_tensors="pt")["pixel_values"]

    return {
        "input_ids": tok["input_ids"],
        "attention_mask": tok["attention_mask"],
        "pixel_values": pixel_values,
        "labels": labels,
        "captions": captions,
    }

neg_map_for_reranker = train_neg_map_semihard if CFG.negatives_for_reranker == "semihard" else train_neg_map_random

train_biencoder_ds = FlickrPairDataset(train_df, mode="biencoder")
val_biencoder_ds = FlickrPairDataset(val_df, mode="biencoder")
test_biencoder_ds = FlickrPairDataset(test_df, mode="biencoder")

train_reranker_ds = FlickrPairDataset(train_df, negative_map=neg_map_for_reranker, mode="reranker")
val_reranker_ds = FlickrPairDataset(val_df, negative_map=build_semihard_negative_map(val_df), mode="reranker")
test_reranker_ds = FlickrPairDataset(test_df, negative_map=build_semihard_negative_map(test_df), mode="reranker")

In [ ]:
train_biencoder_loader = DataLoader(
    train_biencoder_ds,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    collate_fn=collate_biencoder
)

val_biencoder_loader = DataLoader(
    val_biencoder_ds,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    collate_fn=collate_biencoder
)

test_biencoder_loader = DataLoader(
    test_biencoder_ds,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    collate_fn=collate_biencoder
)

train_reranker_loader = DataLoader(
    train_reranker_ds,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    collate_fn=collate_reranker
)

val_reranker_loader = DataLoader(
    val_reranker_ds,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    collate_fn=collate_reranker
)

test_reranker_loader = DataLoader(
    test_reranker_ds,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    collate_fn=collate_reranker
)

#### **8. Modelo A: bi-encoder contrastivo**

El primer modelo del cuaderno adopta una arquitectura de **bi-encoder contrastivo**, en la que cada modalidad se procesa mediante una rama propia: una dedicada al texto y otra a la imagen. A diferencia de los esquemas de fusión profunda, aquí no se modela inicialmente la interacción detallada entre ambas entradas; en su lugar, cada modalidad es transformada de manera independiente y posteriormente proyectada a un **espacio de representación compartido**. El objetivo del entrenamiento consiste en aproximar los pares imagen–texto correctos y alejar aquellos que no corresponden, utilizando para ello un criterio contrastivo aplicado a nivel de batch.


El bi-encoder parte de tres supuestos:

* el texto y la imagen pueden ser codificados por separado sin perder la posibilidad de establecer correspondencias significativas,
* ambas modalidades pueden proyectarse a una geometría común en la que la cercanía represente compatibilidad semántica,
* el contraste entre pares correctos e incorrectos permite aprender un espacio útil para tareas de recuperación multimodal.

En este sentido, el modelo no busca capturar toda la complejidad de la interacción cruzada desde el inicio, sino construir una representación compartida que funcione como base para operaciones posteriores de búsqueda, selección o refinamiento.

El interés de esta arquitectura radica en varias propiedades que la vuelven especialmente relevante en contextos de investigación aplicada y sistemas escalables.

**Retrieval eficiente.**
Al codificar texto e imagen por separado, es posible precalcular embeddings y reutilizarlos en procesos de búsqueda. Esto vuelve al modelo especialmente adecuado para tareas de recuperación a gran escala.

**Embeddings reutilizables.**
Las representaciones producidas por cada torre no dependen de un emparejamiento específico en tiempo de inferencia, lo que permite emplearlas en distintos escenarios downstream, como ranking inicial, clustering o análisis de similitud.

**Separación clara entre búsqueda y refinamiento.**
El bi-encoder permite distinguir con claridad dos fases del sistema: una fase inicial de alineamiento y recuperación, y una fase posterior de re-ranking o verificación fina. Esta separación no solo tiene ventajas computacionales, sino también valor analítico, porque permite estudiar experimentalmente qué parte de la mejora proviene del espacio compartido y cuál del refinamiento posterior.

In [ ]:
class TextTower(nn.Module):
    def __init__(self, model_name: str, embed_dim: int, dropout: float = 0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.proj = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, embed_dim)
        )

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]
        z = self.proj(cls)
        return F.normalize(z, dim=-1)

class ImageTower(nn.Module):
    def __init__(self, embed_dim: int, dropout: float = 0.1):
        super().__init__()
        self.backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        in_dim = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        self.proj = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_dim, embed_dim)
        )

    def forward(self, pixel_values):
        feats = self.backbone(pixel_values)
        z = self.proj(feats)
        return F.normalize(z, dim=-1)

class ContrastiveBiEncoder(nn.Module):
    def __init__(self, text_model_name: str, embed_dim: int, dropout: float = 0.1, temperature: float = 0.07):
        super().__init__()
        self.text_tower = TextTower(text_model_name, embed_dim, dropout)
        self.image_tower = ImageTower(embed_dim, dropout)
        self.temperature = temperature

    def forward(self, input_ids, attention_mask, pixel_values):
        z_text = self.text_tower(input_ids, attention_mask)
        z_image = self.image_tower(pixel_values)
        logits = z_image @ z_text.T / self.temperature
        return z_image, z_text, logits

def contrastive_loss(logits):
    labels = torch.arange(logits.size(0), device=logits.device)
    loss_i2t = F.cross_entropy(logits, labels)
    loss_t2i = F.cross_entropy(logits.T, labels)
    return 0.5 * (loss_i2t + loss_t2i)

#### **9. Modelo B: cross-reranker ligero**

El segundo componente del cuaderno introduce una arquitectura de **re-ranking con interacción cruzada**, pensada para operar **después** de la etapa inicial de retrieval. A diferencia del bi-encoder, que construye representaciones separadas de imagen y texto en un espacio compartido, este módulo recibe directamente un **par candidato** —una imagen y un caption— y estima de manera más fina si ambos constituyen una correspondencia válida.

Desde una perspectiva conceptual, este modelo no reemplaza al bi-encoder, sino que cumple una función complementaria. El bi-encoder resuelve el problema de manera eficiente a gran escala: organiza el espacio multimodal, aproxima pares correctos y permite recuperar rápidamente candidatos plausibles. El reranker, en cambio, actúa sobre un conjunto mucho más pequeño de pares ya seleccionados y se concentra en los casos donde la similitud global no basta para resolver la ambigüedad.

El supuesto central del reranker es que no todos los errores de retrieval provienen de una mala representación global. En muchos casos, el problema aparece porque varios captions o varias imágenes comparten rasgos semánticos generales y solo una inspección más detallada de la interacción entre ambas modalidades permite decidir correctamente. Por eso, esta segunda etapa incorpora una forma de **comparación más rica y localizada**, en la que el modelo ya no trabaja solo con embeddings independientes, sino con una representación conjunta del par candidato.

#### **Función dentro del pipeline**

La lógica del pipeline queda, entonces, organizada en dos niveles:

**Primera etapa: búsqueda rápida.**
El bi-encoder produce una recuperación inicial eficiente, permitiendo seleccionar un conjunto reducido de candidatos relevantes.

**Segunda etapa: corrección de ambigüedades.**
El reranker reevalúa esos candidatos desde una perspectiva más fina y decide cuáles de ellos merecen ocupar las posiciones superiores del ranking final.


La principal aportación de este módulo es mostrar que en multimodalidad no siempre conviene enfrentar eficiencia y precisión como si fueran excluyentes. Más bien, puede ser útil distribuir el problema entre una etapa de **alineamiento global** y una etapa posterior de **refinamiento cruzado**. Este patrón es especialmente importante en sistemas modernos de retrieval multimodal, donde la recuperación inicial debe ser escalable, pero la decisión final requiere mayor sensibilidad semántica.



In [ ]:
class LightCrossReranker(nn.Module):
    def __init__(self, text_model_name: str, dropout: float = 0.1):
        super().__init__()
        self.text_encoder = AutoModel.from_pretrained(text_model_name)
        hidden_size = self.text_encoder.config.hidden_size

        self.image_encoder = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        in_dim = self.image_encoder.classifier[1].in_features
        self.image_encoder.classifier = nn.Identity()

        self.image_proj = nn.Linear(in_dim, hidden_size)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=8,
            dim_feedforward=4 * hidden_size,
            dropout=dropout,
            batch_first=True
        )
        self.fusion = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.cls = nn.Linear(hidden_size, 2)

    def forward(self, input_ids, attention_mask, pixel_values):
        text_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_tokens = text_out.last_hidden_state

        img_feats = self.image_encoder(pixel_values)
        img_token = self.image_proj(img_feats).unsqueeze(1)

        fused = torch.cat([img_token, text_tokens], dim=1)
        fused = self.fusion(fused)
        pooled = fused[:, 0]
        logits = self.cls(pooled)
        return logits

#### **10. Utilidades de entrenamiento y evaluación**

In [ ]:
def move_to_device(batch, device):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}

def train_biencoder(model, loader, optimizer, device):
    model.train()
    losses = []
    for batch in tqdm(loader, desc="Train bi-encoder"):
        batch = move_to_device(batch, device)
        optimizer.zero_grad()

        _, _, logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            pixel_values=batch["pixel_values"]
        )
        loss = contrastive_loss(logits)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    return float(np.mean(losses))

@torch.no_grad()
def encode_split(model, loader, device):
    model.eval()
    all_img = []
    all_txt = []
    all_caps = []

    for batch in tqdm(loader, desc="Encode split"):
        batch = move_to_device(batch, device)
        z_img, z_txt, _ = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            pixel_values=batch["pixel_values"]
        )
        all_img.append(z_img.cpu())
        all_txt.append(z_txt.cpu())
        all_caps.extend(batch["captions"])

    return torch.cat(all_img, dim=0), torch.cat(all_txt, dim=0), all_caps

@torch.no_grad()
def retrieval_metrics(z_img, z_txt, ks=(1,5,10)):
    sim = z_img @ z_txt.T
    labels = torch.arange(sim.size(0))

    results = {}
    for k in ks:
        topk_i2t = sim.topk(k=min(k, sim.size(1)), dim=1).indices
        r_i2t = (topk_i2t == labels.unsqueeze(1)).any(dim=1).float().mean().item()

        topk_t2i = sim.T.topk(k=min(k, sim.size(0)), dim=1).indices
        r_t2i = (topk_t2i == labels.unsqueeze(1)).any(dim=1).float().mean().item()

        results[f"i2t_R@{k}"] = r_i2t
        results[f"t2i_R@{k}"] = r_t2i
    return results

def train_reranker(model, loader, optimizer, device):
    model.train()
    losses = []
    y_true = []
    y_pred = []

    for batch in tqdm(loader, desc="Train reranker"):
        batch = move_to_device(batch, device)
        optimizer.zero_grad()
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            pixel_values=batch["pixel_values"]
        )
        loss = F.cross_entropy(logits, batch["labels"])
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        y_true.extend(batch["labels"].cpu().tolist())
        y_pred.extend(logits.argmax(dim=1).cpu().tolist())

    return {
        "loss": float(np.mean(losses)),
        "acc": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
    }

@torch.no_grad()
def eval_reranker(model, loader, device):
    model.eval()
    losses = []
    y_true = []
    y_pred = []

    for batch in tqdm(loader, desc="Eval reranker"):
        batch = move_to_device(batch, device)
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            pixel_values=batch["pixel_values"]
        )
        loss = F.cross_entropy(logits, batch["labels"])
        losses.append(loss.item())
        y_true.extend(batch["labels"].cpu().tolist())
        y_pred.extend(logits.argmax(dim=1).cpu().tolist())

    return {
        "loss": float(np.mean(losses)),
        "acc": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "report": classification_report(y_true, y_pred, output_dict=False),
        "cm": confusion_matrix(y_true, y_pred),
    }

#### **11. Experimento A: entrenamiento del bi-encoder**

In [ ]:
if CFG.run_biencoder:
    biencoder = ContrastiveBiEncoder(
        text_model_name=CFG.text_model_name,
        embed_dim=CFG.embed_dim,
        dropout=CFG.dropout,
        temperature=CFG.temperature
    ).to(DEVICE)

    opt_bi = torch.optim.AdamW(
        biencoder.parameters(),
        lr=CFG.lr_biencoder,
        weight_decay=CFG.weight_decay
    )

    bi_history = []
    for epoch in range(CFG.epochs_biencoder):
        loss = train_biencoder(biencoder, train_biencoder_loader, opt_bi, DEVICE)
        bi_history.append(loss)
        print(f"Epoch {epoch+1}/{CFG.epochs_biencoder} - train loss: {loss:.4f}")
else:
    print("Bi-encoder desactivado.")

In [ ]:
if CFG.run_biencoder:
    val_img, val_txt, val_caps = encode_split(biencoder, val_biencoder_loader, DEVICE)
    val_metrics = retrieval_metrics(val_img, val_txt)
    print("Validación:", val_metrics)

    test_img, test_txt, test_caps = encode_split(biencoder, test_biencoder_loader, DEVICE)
    test_metrics = retrieval_metrics(test_img, test_txt)
    print("Test:", test_metrics)

#### **12. Experimento B: entrenamiento del reranker**

In [ ]:
if CFG.run_reranker:
    reranker = LightCrossReranker(
        text_model_name=CFG.text_model_name,
        dropout=CFG.dropout
    ).to(DEVICE)

    opt_rr = torch.optim.AdamW(
        reranker.parameters(),
        lr=CFG.lr_reranker,
        weight_decay=CFG.weight_decay
    )

    rr_history = []
    for epoch in range(CFG.epochs_reranker):
        train_stats = train_reranker(reranker, train_reranker_loader, opt_rr, DEVICE)
        val_stats = eval_reranker(reranker, val_reranker_loader, DEVICE)
        rr_history.append({"train": train_stats, "val": val_stats})

        print(f"Epoch {epoch+1}/{CFG.epochs_reranker}")
        print("train:", train_stats)
        print("val:", {k: v for k, v in val_stats.items() if k not in ['report', 'cm']})
else:
    print("Reranker desactivado.")

In [ ]:
if CFG.run_reranker:
    test_rr_stats = eval_reranker(reranker, test_reranker_loader, DEVICE)
    print("Test reranker:")
    print({k: v for k, v in test_rr_stats.items() if k not in ['report', 'cm']})
    print()
    print(test_rr_stats["report"])
    print(test_rr_stats["cm"])

#### **13. Pipeline completo: recuperación top-k y re-ranking**

En esta sección se integra la arquitectura completa del cuaderno, articulando en una sola secuencia operativa los dos niveles de decisión que se han venido desarrollando por separado. El objetivo no es únicamente mostrar que el bi-encoder y el reranker pueden coexistir, sino hacer explícita la lógica metodológica que justifica su combinación dentro de un mismo sistema multimodal.

La primera etapa corresponde al **bi-encoder**, cuya función es producir una representación compartida de texto e imagen y, a partir de ella, generar un **ranking inicial de candidatos**. En esta fase, la prioridad es la eficiencia: el modelo debe ser capaz de recorrer un espacio amplio de posibles correspondencias y recuperar con rapidez aquellos elementos que parecen más próximos en el espacio semántico aprendido.

La segunda etapa introduce una restricción deliberada: en lugar de reevaluar todo el espacio de búsqueda, se selecciona únicamente un conjunto reducido de candidatos, definido por el **top-k** del ranking inicial. Esta decisión refleja un principio central en sistemas modernos de recuperación multimodal: la interacción fina entre modalidades no necesita aplicarse exhaustivamente sobre todos los pares posibles, sino solo sobre aquellos casos que ya han sido identificados como plausibles por una etapa previa de alineamiento.

La tercera etapa corresponde al **re-ranking**, donde el reranker reorganiza ese subconjunto top-k a partir de una evaluación más precisa del par imagen–texto. En este punto, el sistema deja de depender únicamente de la similitud global entre embeddings y pasa a considerar una interacción más rica entre ambas modalidades, lo que permite corregir ambigüedades, refinar el ordenamiento y mejorar la calidad final de la recuperación.

Este pipeline constituye el núcleo conceptual del cuaderno porque materializa una distinción fundamental entre dos tipos de operaciones:

* una operación de **búsqueda eficiente**, basada en alineamiento representacional,
* y una operación de **refinamiento semántico**, basada en interacción más detallada.

Esta separación es especialmente importante porque permite estudiar de forma controlada el compromiso entre **escalabilidad** y **precisión**. El bi-encoder resuelve el problema de cobertura y velocidad y el reranker atiende el problema de ambigüedad y granularidad semántica. Juntos forman un sistema más realista y analíticamente más elaborada que cualquiera de las dos partes por separado.



In [ ]:
@torch.no_grad()
def rerank_topk_for_queries(
    biencoder,
    reranker,
    query_texts: List[str],
    candidate_images: List[Any],
    topk: int,
    device
):
    # codificar textos de consulta
    tok = tokenizer(
        query_texts,
        padding=True,
        truncation=True,
        max_length=CFG.max_text_len,
        return_tensors="pt"
    )
    tok = {k: v.to(device) for k, v in tok.items()}

    biencoder.eval()
    reranker.eval()

    z_text = biencoder.text_tower(tok["input_ids"], tok["attention_mask"])

    # codificar imágenes candidatas en batch
    pixel_values = image_processor(images=candidate_images, return_tensors="pt")["pixel_values"].to(device)
    z_images = biencoder.image_tower(pixel_values)

    sim = z_text @ z_images.T
    topk_idx = sim.topk(k=min(topk, z_images.size(0)), dim=1).indices

    reranked = []
    for qi, idxs in enumerate(topk_idx):
        sub_images = [candidate_images[j] for j in idxs.cpu().tolist()]
        sub_texts = [query_texts[qi]] * len(sub_images)

        tok_rr = tokenizer(
            sub_texts,
            padding=True,
            truncation=True,
            max_length=CFG.max_text_len,
            return_tensors="pt"
        )
        tok_rr = {k: v.to(device) for k, v in tok_rr.items()}
        pix_rr = image_processor(images=sub_images, return_tensors="pt")["pixel_values"].to(device)

        logits = reranker(
            input_ids=tok_rr["input_ids"],
            attention_mask=tok_rr["attention_mask"],
            pixel_values=pix_rr
        )
        scores = F.softmax(logits, dim=-1)[:, 1]
        order = torch.argsort(scores, descending=True)
        reranked_idxs = idxs[order].cpu().tolist()

        reranked.append({
            "initial_topk": idxs.cpu().tolist(),
            "reranked_topk": reranked_idxs,
        })
    return reranked

In [ ]:
# Demo corto del pipeline completo
if CFG.run_biencoder and CFG.run_reranker:
    candidate_pool = list(val_df.groupby("image_id").first()["image"].values)[:50]
    query_pool = val_df["caption"].tolist()[:5]

    pipeline_demo = rerank_topk_for_queries(
        biencoder=biencoder,
        reranker=reranker,
        query_texts=query_pool,
        candidate_images=candidate_pool,
        topk=CFG.topk_for_reranking,
        device=DEVICE
    )
    pipeline_demo[:2]

#### **14. Ejemplos cualitativos de retrieval**

In [ ]:
@torch.no_grad()
def inspect_retrieval_examples(z_img, z_txt, captions, num_examples=5):
    sim = z_img @ z_txt.T
    rows = []
    for i in range(min(num_examples, sim.size(0))):
        ranking = sim[i].topk(k=min(10, sim.size(1))).indices.tolist()
        correct_rank = ranking.index(i) + 1 if i in ranking else None
        rows.append({
            "query_index": i,
            "caption": captions[i],
            "top10_indices": ranking,
            "correct_rank_in_top10": correct_rank
        })
    return pd.DataFrame(rows)

if CFG.run_biencoder:
    inspect_df = inspect_retrieval_examples(test_img, test_txt, test_caps, num_examples=5)
    inspect_df

#### **15. Tabla de resultados**

In [ ]:
results_table = []

if CFG.run_biencoder:
    row = {"modelo": "Bi-encoder contrastivo"}
    row.update({k: round(v, 4) for k, v in test_metrics.items()})
    results_table.append(row)

if CFG.run_reranker:
    row = {"modelo": "Cross-reranker"}
    row.update({
        "loss": round(test_rr_stats["loss"], 4),
        "acc": round(test_rr_stats["acc"], 4),
        "f1": round(test_rr_stats["f1"], 4),
    })
    results_table.append(row)

results_df = pd.DataFrame(results_table)
results_df

#### **16. Ablaciones sugeridas**

Este cuaderno adquiere una densidad analítica mucho mayor cuando no se limita a ejecutar un único pipeline, sino que incorpora al menos una **comparación controlada entre variantes**. En ese sentido, las ablacionesno deben entenderse como un añadido opcional meramente técnico, sino como el mecanismo experimental que permite aislar qué componente del sistema explica una eventual mejora en desempeño.

La función metodológica de una ablación es sencilla pero fundamental: mantener constante la mayor cantidad posible de factores y modificar solo un componente relevante del modelo o del procedimiento, de modo que la diferencia observada en las métricas pueda interpretarse de manera defendible. En este cuaderno, las comparaciones más naturales se organizan alrededor de cuatro ejes.

**Ablación A: negativos aleatorios frente a negativos semi-duros**

Esta comparación examina el papel de la **dificultad del negativo** en el aprendizaje contrastivo y en la posterior etapa de matching. La pregunta subyacente es si el modelo aprende representaciones más informativas cuando se enfrenta a ejemplos incorrectos triviales o cuando debe distinguir entre candidatos semánticamente cercanos pero aún incorrectos.

Desde el punto de vista experimental, esta ablación permite poner a prueba una hipótesis central del cuaderno: que la calidad del alineamiento no depende únicamente de la arquitectura, sino también de la estructura de los contrastes sobre los que esa arquitectura es entrenada.

**Ablación B: sin re-ranking frente a re-ranking sobre top-k**

Esta comparación analiza el valor añadido de una segunda etapa de interacción cruzada después del retrieval inicial. El contraste aquí no es solo entre dos configuraciones de rendimiento, sino entre dos filosofías de sistema: una basada exclusivamente en **alineamiento global eficiente** y otra basada en **alineamiento más refinamiento local**.

La relevancia de esta ablación reside en que permite medir con claridad si el reranker corrige errores reales del ranking inicial o si su aporte es marginal frente al costo computacional adicional que introduce.

**Ablación C: dimensión de embedding reducida frente a dimensión mayor**

Esta comparación se centra en la **capacidad representacional** del espacio compartido. Variar `embed_dim` permite observar hasta qué punto el desempeño depende de la riqueza geométrica del embedding y hasta qué punto el sistema puede seguir funcionando con representaciones más compactas.

Metodológicamente, esta ablación es importante porque ayuda a separar dos explicaciones posibles de una mejora: una basada en mejor diseño del pipeline y otra basada simplemente en mayor capacidad paramétrica.

**Ablación D: backbone visual ligero frente a backbone visual más fuerte**

Esta comparación examina cuánto del resultado final depende del módulo visual empleado. En términos de interpretación, resulta especialmente útil para distinguir si el comportamiento del sistema está dominado por el mecanismo de alineamiento y reranking o si, por el contrario, una parte sustancial de la mejora proviene del encoder visual subyacente.

En contextos de investigación, esta ablación es clave porque permite responder a una objeción frecuente: si una variante funciona mejor, ¿se debe realmente al método propuesto o solo a que dispone de un extractor visual más potente?.




In [ ]:
def make_reranker_loader(df, negative_strategy="semihard"):
    if negative_strategy == "random":
        neg_map = build_random_negative_map(df)
    elif negative_strategy == "semihard":
        neg_map = build_semihard_negative_map(df)
    else:
        raise ValueError("negative_strategy debe ser 'random' o 'semihard'")

    ds = FlickrPairDataset(df, negative_map=neg_map, mode="reranker")
    return DataLoader(
        ds,
        batch_size=CFG.batch_size,
        shuffle=True,
        num_workers=CFG.num_workers,
        collate_fn=collate_reranker
    )

ablación_TEMPLATE = {
    "comparison": "",
    "baseline": "",
    "variant": "",
    "main_metric": "",
    "observed_difference": "",
    "interpretation": "",
}
ablación_TEMPLATE

#### **17. Preguntas de discusión**

1. ¿Qué gana el bi-encoder en eficiencia frente al reranker?
2. ¿Qué pierde en interacción fina?
3. ¿Qué clase de error corrige mejor el reranker?
4. ¿En qué escenarios bastaría con el bi-encoder?
5. ¿Qué experimentos faltan para afirmar que los negativos semi-duros realmente ayudan?
6. ¿Cómo cambiaría el cuaderno si pasas de imagen-texto a video-texto?
7. ¿Qué parte del pipeline escalaría peor a datos masivos?.